# 7교시. 추출 결과 검증 및 데이터 저장

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/07_validation_export.ipynb)

**이번 교시 행동:** 오류·경고·사람 검토를 분리하고, 공개된 승인 정답 경로에서만 Excel을 만듭니다.

**통과 증거:** `course_outputs/receipt_result.xlsx`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. `TODO`가 있는 셀은 안내된 `None` 또는 짧은 값만 바꿉니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(current, total, title, action, expected, code_help):
    _show_learning_message(
        f"""---
### 🧪 실습 단계 {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 CHECKPOINT와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# 검증 결과와 Excel을 저장할 공통 폴더·파일 인계 함수를 준비합니다. 설정 코드이므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 12, '공통 환경 준비', '검증 결과와 Excel을 저장할 공통 환경을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', '검증 결과와 Excel을 저장할 공통 폴더·파일 인계 함수를 준비합니다. 설정 코드이므로 수정하지 않습니다.')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/document_ai_lecture_2026/"
)

def load_course_assets(*relative_paths):
    if VALIDATION_MODE:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 12, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# 최종 앱에서 사용할 Streamlit 버전을 확인하고 필요할 때만 설치합니다. 버전 숫자는 수업 중 임의로 바꾸지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 12, 'Streamlit 준비', '최종 앱에 필요한 Streamlit 버전을 확인하고 준비합니다.', '오류 없이 끝나면 최종 앱 실행 환경이 준비된 것입니다.', '최종 앱에서 사용할 Streamlit 버전을 확인하고 필요할 때만 설치합니다. 버전 숫자는 수업 중 임의로 바꾸지 않습니다.')

import importlib.metadata
import subprocess

required_streamlit = "1.60.0"
try:
    installed_streamlit = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    installed_streamlit = None
if installed_streamlit != required_streamlit:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"streamlit=={required_streamlit}"]
    )

complete_lab_step(2, 12, '오류 없이 끝나면 최종 앱 실행 환경이 준비된 것입니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `find_spec()`로 openpyxl 설치 여부를 확인합니다. 없을 때만 설치해 Excel 생성 함수를 사용할 수 있게 합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 12, 'Excel 라이브러리 준비', 'Excel 생성과 재검사에 필요한 openpyxl을 준비합니다.', '오류 없이 끝나면 Excel 기능을 사용할 수 있습니다.', '`find_spec()`로 openpyxl 설치 여부를 확인합니다. 없을 때만 설치해 Excel 생성 함수를 사용할 수 있게 합니다.')

import importlib.util
import subprocess
if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpyxl==3.1.5"]
    )
from copy import deepcopy
from datetime import date
from openpyxl import Workbook, load_workbook

complete_lab_step(3, 12, '오류 없이 끝나면 Excel 기능을 사용할 수 있습니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `GOLDEN_RECEIPT`는 검증 규칙과 Excel 구조를 확인할 공개 정답입니다. 이 셀은 데이터를 등록하므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 12, '검증용 정답 데이터 준비', '영수증 원문·품목·원문 근거가 포함된 정답을 등록합니다.', '오류 없이 끝나면 검증용 데이터가 준비된 것입니다.', '`GOLDEN_RECEIPT`는 검증 규칙과 Excel 구조를 확인할 공개 정답입니다. 이 셀은 데이터를 등록하므로 수정하지 않습니다.')

GOLDEN_OCR_TEXT = '이태리집\n거래일시 2025-10-04 12:33:37\n페퍼로니 앤 치즈 29,000 1 29,000\n토마토 파스타 14,000 1 14,000\n수제 돈가스 13,000 1 13,000\n새우 칠리치 필라 14,000 1 14,000\n콜라 2,000 3 6,000\n합계 금액 76,000\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_VLM_MARKDOWN = '# 이태리집\n\n> **PREPARED VLM STRUCTURE FIXTURE** — 현재 실행에서 VLM을 호출한 결과가 아닙니다.\n\n거래일시: 2025-10-04 12:33:37\n\n| 품목 | 수량 | 단가 | 금액 |\n| --- | ---: | ---: | ---: |\n| 페퍼로니 앤 치즈 | 1 | 29,000원 | 29,000원 |\n| 토마토 파스타 | 1 | 14,000원 | 14,000원 |\n| 수제 돈가스 | 1 | 13,000원 | 13,000원 |\n| 새우 칠리치 필라 | 1 | 14,000원 | 14,000원 |\n| 콜라 | 3 | 2,000원 | 6,000원 |\n\n**합계: 76,000원**\n\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_RECEIPT = {'document_type': 'receipt',
 'store_name': '이태리집',
 'date': '2025-10-04',
 'total_amount': 76000,
 'items': [{'name': '페퍼로니 앤 치즈',
            'quantity': 1,
            'unit_price': 29000,
            'line_total': 29000},
           {'name': '토마토 파스타', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000},
           {'name': '수제 돈가스', 'quantity': 1, 'unit_price': 13000, 'line_total': 13000},
           {'name': '새우 칠리치 필라',
            'quantity': 1,
            'unit_price': 14000,
            'line_total': 14000},
           {'name': '콜라', 'quantity': 3, 'unit_price': 2000, 'line_total': 6000}],
 'adjustments': {'discount': 0, 'tax': 0, 'service': 0, 'rounding': 0},
 'tax_breakdown': {'mode': 'included_in_item_prices',
                   'supply_amount': 69094,
                   'vat': 6906,
                   'payable_total': 76000},
 'raw_values': {'store_name': '이태리집',
                'date': '2025-10-04 12:33:37',
                'total_amount': '76,000'},
 'cleaned_values': {'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000},
 'evidence': {'store_name': {'raw_value': '이태리집', 'line': 1},
              'date': {'raw_value': '거래일시 2025-10-04 12:33:37', 'line': 2},
              'total_amount': {'raw_value': '합계 금액 76,000', 'line': 8}},
 'source_mode': 'prepared_fixture_rule_extraction'}

complete_lab_step(4, 12, '오류 없이 끝나면 검증용 데이터가 준비된 것입니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `USE_PREPARED_INPUT=True`는 공개 입력, `False`는 4교시 JSON 업로드입니다. `validate_receipt()`
# 결과의 오류와 경고를 확인합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 12, '이전 교시 결과 불러오기', '이전 JSON을 읽거나 공개 복구 입력으로 전환합니다.', '입력 모드와 검증 결과가 표시되어야 합니다.', '`USE_PREPARED_INPUT=True`는 공개 입력, `False`는 4교시 JSON 업로드입니다. `validate_receipt()` 결과의 오류와 경고를 확인합니다.')

input_path = OUTPUT_DIR / "receipt.json"
# 기본값 True: 새 Colab에서도 공개 준비 입력으로 바로 실행합니다.
# 앞 교시 파일을 이어 쓰려면 False로 바꾸고 업로드 창에서 선택합니다.
USE_PREPARED_INPUT = True
if not input_path.exists() and not USE_PREPARED_INPUT:
    upload_previous_artifact("receipt.json")
if input_path.exists():
    receipt = json.loads(input_path.read_text(encoding="utf-8"))
    INPUT_MODE = "PREVIOUS_LESSON"
else:
    receipt = deepcopy(GOLDEN_RECEIPT)
    INPUT_MODE = "PREPARED_FALLBACK"


def validate_receipt(data):
    warnings, errors = [], []
    for field in ("store_name", "date", "total_amount", "items"):
        if data.get(field) in (None, "", []):
            errors.append(f"필수값 누락: {field}")
    try:
        parsed_date = date.fromisoformat(data.get("date", ""))
        if parsed_date > date.today():
            warnings.append("미래 날짜입니다. 원본을 확인하세요.")
    except ValueError:
        errors.append("date는 YYYY-MM-DD 형식이어야 합니다.")

    total = data.get("total_amount")
    if isinstance(total, bool) or not isinstance(total, int) or total < 0:
        errors.append("total_amount는 0 이상의 정수여야 합니다.")
    item_sum = 0
    for index, item in enumerate(data.get("items") or [], start=1):
        values = [item.get(key) for key in ("quantity", "unit_price", "line_total")]
        if not all(isinstance(value, int) and not isinstance(value, bool) for value in values):
            errors.append(f"{index}번째 품목 금액 형식 오류")
            continue
        if values[0] * values[1] != values[2]:
            errors.append(f"{index}번째 품목 수량×단가 오류")
        item_sum += values[2]
    adjustments = data.get("adjustments") or {}
    expected = (
        item_sum
        - adjustments.get("discount", 0)
        + adjustments.get("tax", 0)
        + adjustments.get("service", 0)
        + adjustments.get("rounding", 0)
    )
    if isinstance(total, int) and not isinstance(total, bool) and expected != total:
        errors.append(f"품목·조정 후 합계 {expected:,}원과 총액 {total:,}원이 다릅니다.")
    tax_breakdown = data.get("tax_breakdown")
    if tax_breakdown and tax_breakdown.get("mode") == "included_in_item_prices":
        supply = tax_breakdown.get("supply_amount")
        vat = tax_breakdown.get("vat")
        payable = tax_breakdown.get("payable_total")
        if not all(isinstance(value, int) and not isinstance(value, bool)
                   for value in (supply, vat, payable)):
            errors.append("포함세액 내역은 정수 금액이어야 합니다.")
        elif supply + vat != payable or payable != total:
            errors.append("공급가액·포함 부가세·총액 관계가 맞지 않습니다.")
        if adjustments.get("tax", 0) != 0:
            errors.append("포함 부가세를 adjustments.tax에 다시 더하면 이중 계산됩니다.")
    for field in ("store_name", "date", "total_amount"):
        if not (data.get("evidence") or {}).get(field):
            warnings.append(f"{field}의 원본 근거가 없습니다.")
    return {"valid": not errors, "warnings": warnings, "errors": errors}


validation = validate_receipt(receipt)
source_text = receipt.get("source_text") or GOLDEN_OCR_TEXT
print("입력 모드:", INPUT_MODE)
print("검증:", validation)
if not validation["valid"]:
    print(
        "BLOCKED_BY_VALIDATION: 아래 최종 앱에서 원본과 대조해 "
        "값을 수정한 뒤 승인하세요."
    )

complete_lab_step(5, 12, '입력 모드와 검증 결과가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `validate_receipt()`는 필수값·계산·근거를 검사하고 `export_excel()`은 검증과 승인 조건이 모두 맞을 때만 세 시트를
# 만듭니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 12, '검증·Excel 함수 준비', '오류·경고·승인 상태를 검사하고 Excel을 만드는 함수를 등록합니다.', '오류 없이 끝나면 검증과 저장 함수를 사용할 수 있습니다.', '`validate_receipt()`는 필수값·계산·근거를 검사하고 `export_excel()`은 검증과 승인 조건이 모두 맞을 때만 세 시트를 만듭니다.')

def safe_text(value):
    if isinstance(value, str) and value.lstrip(" \t\r\n").startswith(
        ("=", "+", "-", "@")
    ):
        return "'" + value
    return value


def save_reviewed_excel(data, validation, review_record, output_path, source_text):
    if not validation["valid"]:
        return False
    if review_record.get("decision") not in {"APPROVED", "CHANGED"}:
        return False

    workbook = Workbook()
    summary = workbook.active
    summary.title = "검토_요약"
    summary.append([
        "field", "raw_value", "cleaned_value", "final_value",
        "decision", "reviewer", "reviewed_at", "change_reason",
    ])
    raw = data.get("raw_values") or {}
    cleaned = data.get("cleaned_values") or {}
    for field in ("store_name", "date", "total_amount"):
        summary.append([
            field,
            safe_text(raw.get(field)),
            safe_text(cleaned.get(field)),
            safe_text(data.get(field)),
            review_record["decision"],
            safe_text(review_record["reviewer"]),
            review_record["reviewed_at"],
            safe_text(review_record["note"]),
        ])

    items = workbook.create_sheet("품목")
    items.append(["품목", "수량", "단가", "금액"])
    for item in data["items"]:
        items.append([
            safe_text(item["name"]),
            item["quantity"],
            item["unit_price"],
            item["line_total"],
        ])

    evidence = workbook.create_sheet("원문_근거")
    evidence.append(["source_mode", data.get("source_mode")])
    evidence.append(["ocr_text", safe_text(source_text)])
    evidence.append(["evidence", safe_text(json.dumps(
        data.get("evidence") or {}, ensure_ascii=False
    ))])
    workbook.save(output_path)
    return True

complete_lab_step(6, 12, '오류 없이 끝나면 검증과 저장 함수를 사용할 수 있습니다.')


## 시나리오 A. 기본값은 차단

사람이 원본을 보기 전에는 결과가 유효해도 다운로드를 열지 않습니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `PENDING_REVIEW` 상태로 `export_excel()`을 호출합니다. 반환값이 `False`이고 파일이 없으면 미승인 저장 차단이
# 정상입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(7, 12, '미승인 저장 차단 확인', '검토하지 않은 결과로 Excel이 생성되지 않는지 검사합니다.', '`DEFAULT_BLOCKED PASS`와 파일 미생성을 확인합니다.', '`PENDING_REVIEW` 상태로 `export_excel()`을 호출합니다. 반환값이 `False`이고 파일이 없으면 미승인 저장 차단이 정상입니다.')

blocked_path = OUTPUT_DIR / "pending_review.xlsx"
PENDING_REVIEW = {
    "decision": "PENDING",
    "reviewer": "",
    "reviewed_at": "",
    "note": "",
}
assert not save_reviewed_excel(
    receipt, validation, PENDING_REVIEW, blocked_path, source_text
)
assert not blocked_path.exists()
print("DEFAULT_BLOCKED PASS: 미승인 Excel 없음")

complete_lab_step(7, 12, '`DEFAULT_BLOCKED PASS`와 파일 미생성을 확인합니다.')


## 시나리오 B. 내가 직접 남기는 승인 기록

원본의 상호명·날짜·품목·총액을 직접 대조한 뒤 세 곳을 채웁니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `my_source_checked`, `my_decision`, `my_reviewer` 세 곳만 입력합니다. 원본 대조를 끝낸 뒤 실제 검토
# 기록을 남기는 단계입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(8, 12, '내 승인 기록 입력', '원본 대조 여부·결정·검토자 이름을 직접 입력합니다.', '빈칸 안내 또는 내가 남긴 승인 기록이 표시되어야 합니다.', '`my_source_checked`, `my_decision`, `my_reviewer` 세 곳만 입력합니다. 원본 대조를 끝낸 뒤 실제 검토 기록을 남기는 단계입니다.')

# TODO: 원본 대조 뒤 세 곳을 채우세요.
my_decision = None
my_reviewer = None
my_review_note = None
if None in (my_decision, my_reviewer, my_review_note):
    print("빈칸이 있습니다. 아래 전체 정답과 비교하세요.")

complete_lab_step(8, 12, '빈칸 안내 또는 내가 남긴 승인 기록이 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

값 수정이 없으면 `APPROVED`, 수정했다면 `CHANGED`입니다. 검토자와
무엇을 확인했는지도 기록합니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `REVIEW_RECORD`가 공개 승인 정답을 보완하고 Excel을 생성합니다. 저장된 시트 이름과 승인 상태를 다시 열어 검사합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(9, 12, '승인 후 Excel 생성', '공개 승인 경로로 재검증하고 Excel 세 시트를 생성합니다.', '`REVIEWED_APPROVED PASS`와 Excel 파일 경로를 확인합니다.', '`REVIEW_RECORD`가 공개 승인 정답을 보완하고 Excel을 생성합니다. 저장된 시트 이름과 승인 상태를 다시 열어 검사합니다.')

REVIEW_RECORD = {
    "decision": my_decision or "APPROVED",
    "reviewer": my_reviewer or "learner",
    "reviewed_at": "2026-07-28T15:30:00+09:00",
    "note": (
        my_review_note
        or "공개 비식별 원본과 상호명·날짜·품목·총액 대조 완료"
    ),
}
output_path = OUTPUT_DIR / "receipt_result.xlsx"
excel_created = save_reviewed_excel(
    receipt, validation, REVIEW_RECORD, output_path, source_text
)
if excel_created:
    saved = load_workbook(output_path)
    assert saved.sheetnames == ["검토_요약", "품목", "원문_근거"]
    assert saved["검토_요약"]["E2"].value in {"APPROVED", "CHANGED"}
    print("REVIEWED_APPROVED PASS:", output_path, saved.sheetnames)
    print("CHECKPOINT 1/1 PASS: 미승인 차단 + 승인 후 Excel")
    download_artifact(output_path)
else:
    print(
        "Excel 생성 차단: 검증 오류를 최종 앱에서 수정한 뒤 "
        "다운로드하세요."
    )

complete_lab_step(9, 12, '`REVIEWED_APPROVED PASS`와 Excel 파일 경로를 확인합니다.')


## 최종 앱: 업로드부터 Excel 다운로드까지 한 화면으로 연결

아래 셀은 앞 교시의 기능을 하나의 실행 가능한 앱으로 묶습니다.
앱에서는 OCR/VLM 경로 선택, 원문·JSON 확인, 상호명·날짜·총액·품목
수정, 재검증, 사람 승인, Excel 다운로드를 순서대로 수행합니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `packaged_sources`는 최종 앱의 여러 Python 파일입니다. 반복문이 폴더를 만들고 `make_archive()`가 전체 코드를
# ZIP으로 묶습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(10, 12, '최종 앱 묶음 생성', '업로드부터 Excel 다운로드까지 연결된 앱을 ZIP으로 묶습니다.', '최종 앱 경로와 ZIP 파일 경로가 표시되어야 합니다.', '`packaged_sources`는 최종 앱의 여러 Python 파일입니다. 반복문이 폴더를 만들고 `make_archive()`가 전체 코드를 ZIP으로 묶습니다.')

import shutil

packaged_sources = {
    'app.py': (
        '"""Streamlit Document AI 미니 앱.\n'
        '\n'
        '학습자는 Colab에서 이 파일을 만들고 Streamlit AppTest로 기능을 확인한다.\n'
        '"""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'from copy import deepcopy\n'
        'from numbers import Integral, Real\n'
        'import sys\n'
        'from pathlib import Path\n'
        'from tempfile import NamedTemporaryFile\n'
        'from datetime import datetime, timezone\n'
        '\n'
        'import pandas as pd\n'
        'import streamlit as st\n'
        '\n'
        'APP_ROOT = Path(__file__).resolve().parent\n'
        'if str(APP_ROOT) not in sys.path:\n'
        '    sys.path.insert(0, str(APP_ROOT))\n'
        '\n'
        'from src.export import receipt_to_rows, receipt_to_xlsx_bytes\n'
        'from src.pipeline import process_document\n'
        'from src.validate import validate_receipt\n'
        '\n'
        '\n'
        'st.set_page_config(page_title="영수증 Document AI", layout="wide")\n'
        'st.title("영수증 Document AI 미니 앱")\n'
        'st.caption("한 번에 문서 한 장 · 원문 대조 후 Excel 저장")\n'
        'st.warning(\n'
        '    "Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는 "\n'
        '    "업로드하지 마세요. 필수 실습은 공개·합성 샘플로 진행합니다."\n'
        ')\n'
        '\n'
        '\n'
        'def editor_integer(value):\n'
        '    """표 편집값을 정수일 때만 변환하고 소수 입력은 검증기에 남긴다."""\n'
        '\n'
        '    if pd.isna(value):\n'
        '        return None\n'
        '    if isinstance(value, bool):\n'
        '        return value\n'
        '    if isinstance(value, Integral):\n'
        '        return int(value)\n'
        '    if isinstance(value, Real) and float(value).is_integer():\n'
        '        return int(value)\n'
        '    return value\n'
        '\n'
        'uploaded_file = st.file_uploader(\n'
        '    "영수증 이미지 또는 PDF 한 장 · 최대 5MB",\n'
        '    type=["png", "jpg", "jpeg", "pdf"],\n'
        '    accept_multiple_files=False,\n'
        '    max_upload_size=5,\n'
        '    help="PNG, JPG, JPEG, PDF만 허용합니다. 수업에서는 한 번에 5MB 이하 한 장만 처리합니다.",\n'
        ')\n'
        'processor = st.radio(\n'
        '    "처리 경로",\n'
        '    options=["ocr", "vlm"],\n'
        '    format_func=lambda value: (\n'
        '        "PaddleOCR · PP-OCRv5 Korean"\n'
        '        if value == "ocr"\n'
        '        else "문서 VLM · 강사 시연/준비 결과"\n'
        '    ),\n'
        ')\n'
        '\n'
        'left, right = st.columns(2)\n'
        'run_uploaded = left.button("업로드 문서 처리", type="primary", key="run_uploaded")\n'
        'run_sample = right.button("준비 결과로 실행", key="run_sample")\n'
        '\n'
        '\n'
        'def _run_uploaded_file() -> dict:\n'
        '    if uploaded_file is None:\n'
        '        return process_document(None, processor=processor)\n'
        '\n'
        '    suffix = Path(uploaded_file.name).suffix.lower()\n'
        '    with NamedTemporaryFile(suffix=suffix) as temp_file:\n'
        '        temp_file.write(uploaded_file.getvalue())\n'
        '        temp_file.flush()\n'
        '        return process_document(temp_file.name, processor=processor)\n'
        '\n'
        '\n'
        'if run_uploaded:\n'
        '    st.session_state["result"] = _run_uploaded_file()\n'
        '    st.session_state["review_complete"] = False\n'
        'elif run_sample:\n'
        '    st.session_state["result"] = process_document(\n'
        '        processor=processor,\n'
        '        use_sample=True,\n'
        '    )\n'
        '    st.session_state["review_complete"] = False\n'
        '\n'
        'result = st.session_state.get("result")\n'
        'if result:\n'
        '    if not result.get("ok"):\n'
        '        st.error(result.get("status", "처리 오류"))\n'
        '        for message in result.get("errors", []):\n'
        '            st.write(f"- {message}")\n'
        '        st.info("원인을 확인하거나 준비 결과로 실행하세요.")\n'
        '    else:\n'
        '        st.success(result["status"])\n'
        '\n'
        '        tab_text, tab_json, tab_table = st.tabs(\n'
        '            ["판독 원문", "구조화 JSON", "품목 표"]\n'
        '        )\n'
        '        with tab_text:\n'
        '            st.text_area("OCR·문서 판독 결과", result["ocr_text"], height=240)\n'
        '        with tab_json:\n'
        '            st.json(result["data"])\n'
        '        with tab_table:\n'
        '            st.dataframe(\n'
        '                pd.DataFrame(receipt_to_rows(result["data"])),\n'
        '                width="stretch",\n'
        '                hide_index=True,\n'
        '            )\n'
        '\n'
        '        st.subheader("원본 대조 후 수정")\n'
        '        st.caption(\n'
        '            "AI 결과를 그대로 승인하지 말고 원본 영수증과 비교해 수정하세요. "\n'
        '            "수정값으로 다시 검증한 뒤 Excel을 만듭니다."\n'
        '        )\n'
        '        reviewed_data = deepcopy(result["data"])\n'
        '        field_left, field_middle, field_right = st.columns(3)\n'
        '        reviewed_data["store_name"] = field_left.text_input(\n'
        '            "상호명",\n'
        '            value=str(reviewed_data.get("store_name") or ""),\n'
        '            key="review_store_name",\n'
        '        ).strip() or None\n'
        '        reviewed_data["date"] = field_middle.text_input(\n'
        '            "날짜 · YYYY-MM-DD",\n'
        '            value=str(reviewed_data.get("date") or ""),\n'
        '            key="review_date",\n'
        '        ).strip() or None\n'
        '        total_text = field_right.text_input(\n'
        '            "총액 · 숫자만",\n'
        '            value=(\n'
        '                str(reviewed_data["total_amount"])\n'
        '                if isinstance(reviewed_data.get("total_amount"), int)\n'
        '                and not isinstance(reviewed_data.get("total_amount"), bool)\n'
        '                else ""\n'
        '            ),\n'
        '            key="review_total_amount",\n'
        '        )\n'
        '        try:\n'
        '            reviewed_data["total_amount"] = int(\n'
        '                total_text.replace(",", "").strip()\n'
        '            )\n'
        '        except ValueError:\n'
        '            reviewed_data["total_amount"] = None\n'
        '\n'
        '        editable_items = pd.DataFrame(\n'
        '            reviewed_data.get("items") or [],\n'
        '            columns=["name", "quantity", "unit_price", "line_total"],\n'
        '        )\n'
        '        edited_items = st.data_editor(\n'
        '            editable_items,\n'
        '            width="stretch",\n'
        '            hide_index=True,\n'
        '            num_rows="dynamic",\n'
        '            key="review_items",\n'
        '            column_config={\n'
        '                "name": st.column_config.TextColumn("품목명"),\n'
        '                "quantity": st.column_config.NumberColumn("수량", min_value=0, step=1),\n'
        '                "unit_price": st.column_config.NumberColumn(\n'
        '                    "단가", min_value=0, step=1\n'
        '                ),\n'
        '                "line_total": st.column_config.NumberColumn(\n'
        '                    "금액", min_value=0, step=1\n'
        '                ),\n'
        '            },\n'
        '        )\n'
        '        reviewed_data["items"] = [\n'
        '            {\n'
        '                "name": str(row.get("name") or "").strip() or None,\n'
        '                "quantity": editor_integer(row.get("quantity")),\n'
        '                "unit_price": editor_integer(row.get("unit_price")),\n'
        '                "line_total": editor_integer(row.get("line_total")),\n'
        '            }\n'
        '            for row in edited_items.to_dict("records")\n'
        '        ]\n'
        '\n'
        '        validation = validate_receipt(reviewed_data)\n'
        '        if validation["valid"]:\n'
        '            st.success("규칙 검증 통과 · 사람 승인 대기")\n'
        '        for warning in validation["warnings"]:\n'
        '            st.warning(warning)\n'
        '        for error in validation["errors"]:\n'
        '            st.error(error)\n'
        '\n'
        '        reviewer = st.text_input(\n'
        '            "검토자 ID 또는 교육용 이름",\n'
        '            value="learner",\n'
        '            key="reviewer",\n'
        '        )\n'
        '        review_note = st.text_input(\n'
        '            "수정·확인 메모",\n'
        '            value="원본의 상호명·날짜·품목·총액 대조 완료",\n'
        '            key="review_note",\n'
        '        )\n'
        '        review_complete = st.checkbox(\n'
        '            "원본 영수증과 추출값을 직접 대조했고 이 결과를 승인합니다.",\n'
        '            key="review_complete",\n'
        '        )\n'
        '        if validation["valid"] and review_complete and reviewer.strip():\n'
        '            decision = (\n'
        '                "CHANGED"\n'
        '                if reviewed_data != result["data"]\n'
        '                else "APPROVED"\n'
        '            )\n'
        '            review_record = {\n'
        '                "decision": decision,\n'
        '                "reviewer": reviewer.strip(),\n'
        '                "reviewed_at": datetime.now(timezone.utc).astimezone().isoformat(\n'
        '                    timespec="seconds"\n'
        '                ),\n'
        '                "note": review_note.strip(),\n'
        '            }\n'
        '            xlsx_bytes = receipt_to_xlsx_bytes(\n'
        '                reviewed_data,\n'
        '                source_text=result["ocr_text"],\n'
        '                review_status=decision,\n'
        '                review_record=review_record,\n'
        '            )\n'
        '            st.download_button(\n'
        '                "검증된 Excel 다운로드",\n'
        '                data=xlsx_bytes,\n'
        '                file_name="receipt_result.xlsx",\n'
        '                mime=(\n'
        '                    "application/vnd.openxmlformats-officedocument."\n'
        '                    "spreadsheetml.sheet"\n'
        '                ),\n'
        '            )\n'
        '        elif validation["errors"]:\n'
        '            st.warning("검증 오류가 남아 있어 Excel 다운로드를 차단했습니다.")\n'
        '        elif review_complete and not reviewer.strip():\n'
        '            st.info("승인 기록에 남길 검토자 ID를 입력하세요.")\n'
        '        else:\n'
        '            st.info("원본을 확인하고 승인해야 Excel을 다운로드할 수 있습니다.")\n'
    ),
    'src/__init__.py': (
        '"""초보자용 Document AI 실습 함수 모음."""\n'
        '\n'
        'from .clean import group_receipt_lines, normalize_line\n'
        'from .export import receipt_to_rows, receipt_to_xlsx_bytes\n'
        'from .extract import RECEIPT_SCHEMA, build_extraction_prompt, mock_extract\n'
        'from .ocr import extract_with_paddleocr, load_mock_ocr, reconstruct_spatial_lines\n'
        'from .pipeline import process_document, run_smoke_test\n'
        'from .validate import validate_receipt\n'
        'from .vlm import load_mock_vlm, parse_with_paddleocr_vl\n'
        '\n'
        '__all__ = [\n'
        '    "RECEIPT_SCHEMA",\n'
        '    "build_extraction_prompt",\n'
        '    "extract_with_paddleocr",\n'
        '    "group_receipt_lines",\n'
        '    "load_mock_ocr",\n'
        '    "load_mock_vlm",\n'
        '    "mock_extract",\n'
        '    "normalize_line",\n'
        '    "process_document",\n'
        '    "parse_with_paddleocr_vl",\n'
        '    "receipt_to_xlsx_bytes",\n'
        '    "receipt_to_rows",\n'
        '    "reconstruct_spatial_lines",\n'
        '    "run_smoke_test",\n'
        '    "validate_receipt",\n'
        ']\n'
    ),
    'src/clean.py': (
        '"""3교시: OCR 원문을 잃지 않고 정리하는 함수."""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import re\n'
        '\n'
        '\n'
        'def normalize_line(line: str) -> tuple[str, list[str]]:\n'
        '    """한 줄의 불필요한 공백만 정리하고 변경 기록을 반환한다."""\n'
        '\n'
        '    original = line\n'
        '    cleaned = line.strip()\n'
        '    cleaned = re.sub(r"\\s+", " ", cleaned)\n'
        '\n'
        '    changes: list[str] = []\n'
        '    if original != cleaned:\n'
        '        changes.append(f"공백 정리: {original!r} → {cleaned!r}")\n'
        '    return cleaned, changes\n'
        '\n'
        '\n'
        'def group_receipt_lines(raw_text: str) -> dict:\n'
        '    """영수증 OCR 텍스트를 헤더·날짜·품목·합계 줄로 분류한다."""\n'
        '\n'
        '    cleaned_lines: list[str] = []\n'
        '    change_log: list[str] = []\n'
        '\n'
        '    for raw_line in raw_text.splitlines():\n'
        '        cleaned, changes = normalize_line(raw_line)\n'
        '        if cleaned:\n'
        '            cleaned_lines.append(cleaned)\n'
        '            change_log.extend(changes)\n'
        '\n'
        '    groups = {\n'
        '        "header": [],\n'
        '        "date": [],\n'
        '        "items": [],\n'
        '        "total": [],\n'
        '        "other": [],\n'
        '    }\n'
        '\n'
        '    for line in cleaned_lines:\n'
        '        if "거래일자" in line:\n'
        '            groups["date"].append(line)\n'
        '        elif "합계" in line:\n'
        '            groups["total"].append(line)\n'
        '        elif "개" in line and ("×" in line or "x" in line.lower()):\n'
        '            groups["items"].append(line)\n'
        '        elif not groups["header"]:\n'
        '            groups["header"].append(line)\n'
        '        else:\n'
        '            groups["other"].append(line)\n'
        '\n'
        '    return {\n'
        '        "raw_text": raw_text,\n'
        '        "cleaned_lines": cleaned_lines,\n'
        '        "groups": groups,\n'
        '        "change_log": change_log,\n'
        '    }\n'
    ),
    'src/export.py': (
        '"""7교시: 검증된 영수증 JSON을 안전한 Excel 파일로 바꾼다."""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import io\n'
        'from typing import Any\n'
        '\n'
        'from openpyxl import Workbook\n'
        'from openpyxl.styles import Font, PatternFill\n'
        'from openpyxl.utils import get_column_letter\n'
        '\n'
        '\n'
        'FORMULA_PREFIXES = ("=", "+", "-", "@")\n'
        '\n'
        '\n'
        'def safe_spreadsheet_text(value: Any) -> Any:\n'
        '    """스프레드시트에서 수식으로 해석될 수 있는 문자열을 보호한다."""\n'
        '\n'
        '    if isinstance(value, str) and value.lstrip(" \\t\\r\\n").startswith(\n'
        '        FORMULA_PREFIXES\n'
        '    ):\n'
        '        return "\'" + value\n'
        '    return value\n'
        '\n'
        '\n'
        'def receipt_to_rows(data: dict) -> list[dict]:\n'
        '    """영수증의 반복 품목을 Excel의 여러 행으로 펼친다."""\n'
        '\n'
        '    base = {\n'
        '        "store_name": data.get("store_name"),\n'
        '        "date": data.get("date"),\n'
        '        "total_amount": data.get("total_amount"),\n'
        '    }\n'
        '    rows = []\n'
        '    for item in data.get("items") or []:\n'
        '        row = {\n'
        '            **base,\n'
        '            "item_name": item.get("name"),\n'
        '            "quantity": item.get("quantity"),\n'
        '            "unit_price": item.get("unit_price"),\n'
        '            "line_total": item.get("line_total"),\n'
        '        }\n'
        '        rows.append(\n'
        '            {key: safe_spreadsheet_text(value) for key, value in row.items()}\n'
        '        )\n'
        '    return rows\n'
        '\n'
        '\n'
        'def _write_table(sheet, columns: list[str], rows: list[dict]) -> None:\n'
        '    """작은 표를 쓰고 초보자용 기본 서식을 적용한다."""\n'
        '\n'
        '    sheet.append(columns)\n'
        '    for cell in sheet[1]:\n'
        '        cell.font = Font(bold=True, color="FFFFFF")\n'
        '        cell.fill = PatternFill("solid", fgColor="173B57")\n'
        '\n'
        '    for row in rows:\n'
        '        sheet.append([safe_spreadsheet_text(row.get(column)) for column in columns])\n'
        '\n'
        '    sheet.freeze_panes = "A2"\n'
        '    for index, column in enumerate(columns, start=1):\n'
        '        values = [str(column)]\n'
        '        values.extend(\n'
        '            str(sheet.cell(row=row, column=index).value or "")\n'
        '            for row in range(2, sheet.max_row + 1)\n'
        '        )\n'
        '        width = min(max(len(value) for value in values) + 2, 40)\n'
        '        sheet.column_dimensions[get_column_letter(index)].width = width\n'
        '\n'
        '\n'
        'def receipt_to_xlsx_bytes(\n'
        '    data: dict,\n'
        '    *,\n'
        '    source_text: str = "",\n'
        '    review_status: str = "APPROVED",\n'
        '    review_record: dict | None = None,\n'
        ') -> bytes:\n'
        '    """원문·정제값·최종값·검토 상태가 남는 Excel 바이트를 반환한다."""\n'
        '\n'
        '    workbook = Workbook()\n'
        '    summary = workbook.active\n'
        '    summary.title = "검토_요약"\n'
        '\n'
        '    raw_values = data.get("raw_values") or {}\n'
        '    cleaned_values = data.get("cleaned_values") or {}\n'
        '    review_record = review_record or {\n'
        '        "decision": review_status,\n'
        '        "reviewer": "교육용 검수자",\n'
        '        "reviewed_at": "",\n'
        '        "note": "",\n'
        '    }\n'
        '    fields = ("store_name", "date", "total_amount")\n'
        '    summary_rows = [\n'
        '        {\n'
        '            "field": field,\n'
        '            "raw_value": raw_values.get(field, data.get(field)),\n'
        '            "cleaned_value": cleaned_values.get(field, data.get(field)),\n'
        '            "final_value": data.get(field),\n'
        '            "decision": review_record.get("decision", review_status),\n'
        '            "reviewer": review_record.get("reviewer", ""),\n'
        '            "reviewed_at": review_record.get("reviewed_at", ""),\n'
        '            "change_reason": review_record.get("note", ""),\n'
        '        }\n'
        '        for field in fields\n'
        '    ]\n'
        '    _write_table(\n'
        '        summary,\n'
        '        [\n'
        '            "field",\n'
        '            "raw_value",\n'
        '            "cleaned_value",\n'
        '            "final_value",\n'
        '            "decision",\n'
        '            "reviewer",\n'
        '            "reviewed_at",\n'
        '            "change_reason",\n'
        '        ],\n'
        '        summary_rows,\n'
        '    )\n'
        '\n'
        '    items = workbook.create_sheet("품목")\n'
        '    item_columns = [\n'
        '        "store_name",\n'
        '        "date",\n'
        '        "total_amount",\n'
        '        "item_name",\n'
        '        "quantity",\n'
        '        "unit_price",\n'
        '        "line_total",\n'
        '    ]\n'
        '    _write_table(items, item_columns, receipt_to_rows(data))\n'
        '\n'
        '    source = workbook.create_sheet("원문_근거")\n'
        '    source.append(["source_mode", safe_spreadsheet_text(data.get("source_mode", ""))])\n'
        '    provenance = data.get("provenance") or {}\n'
        '    for key in (\n'
        '        "fixture_type",\n'
        '        "input_file",\n'
        '        "input_sha256",\n'
        '        "engine",\n'
        '        "engine_version",\n'
        '        "target_technology",\n'
        '        "recorded_at",\n'
        '        "reviewer",\n'
        '        "disclaimer",\n'
        '    ):\n'
        '        source.append([key, safe_spreadsheet_text(provenance.get(key, ""))])\n'
        '    source.append(["ocr_text", safe_spreadsheet_text(source_text)])\n'
        '    source.append(["evidence", safe_spreadsheet_text(str(data.get("evidence", {})))])\n'
        '    source.column_dimensions["A"].width = 18\n'
        '    source.column_dimensions["B"].width = 80\n'
        '\n'
        '    buffer = io.BytesIO()\n'
        '    workbook.save(buffer)\n'
        '    return buffer.getvalue()\n'
    ),
    'src/extract.py': (
        '"""4교시: OCR 텍스트를 영수증 JSON으로 구조화하는 함수."""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import json\n'
        'import re\n'
        'from copy import deepcopy\n'
        'from typing import Any\n'
        '\n'
        'from .sample_data import SAMPLE_RECEIPT\n'
        '\n'
        '\n'
        'RECEIPT_SCHEMA: dict[str, Any] = {\n'
        '    "type": "object",\n'
        '    "required": [\n'
        '        "document_type",\n'
        '        "store_name",\n'
        '        "date",\n'
        '        "total_amount",\n'
        '        "items",\n'
        '    ],\n'
        '    "properties": {\n'
        '        "document_type": {"const": "receipt"},\n'
        '        "store_name": {"type": ["string", "null"]},\n'
        '        "date": {\n'
        '            "type": ["string", "null"],\n'
        '            "description": "YYYY-MM-DD",\n'
        '        },\n'
        '        "total_amount": {"type": ["integer", "null"], "minimum": 0},\n'
        '        "items": {\n'
        '            "type": "array",\n'
        '            "items": {\n'
        '                "type": "object",\n'
        '                "required": ["name", "quantity", "unit_price", "line_total"],\n'
        '                "properties": {\n'
        '                    "name": {"type": ["string", "null"]},\n'
        '                    "quantity": {"type": ["integer", "null"], "minimum": 0},\n'
        '                    "unit_price": {"type": ["integer", "null"], "minimum": 0},\n'
        '                    "line_total": {"type": ["integer", "null"], "minimum": 0},\n'
        '                },\n'
        '            },\n'
        '        },\n'
        '        "adjustments": {\n'
        '            "type": "object",\n'
        '            "properties": {\n'
        '                "discount": {"type": "integer"},\n'
        '                "tax": {"type": "integer"},\n'
        '                "service": {"type": "integer"},\n'
        '                "rounding": {"type": "integer"},\n'
        '            },\n'
        '        },\n'
        '        "tax_breakdown": {\n'
        '            "type": ["object", "null"],\n'
        '            "description": "품목 가격에 이미 포함된 공급가액·부가세 표시",\n'
        '        },\n'
        '        "evidence": {"type": "object"},\n'
        '        "raw_values": {"type": "object"},\n'
        '        "cleaned_values": {"type": "object"},\n'
        '        "provenance": {"type": "object"},\n'
        '        "source_mode": {"type": "string"},\n'
        '    },\n'
        '}\n'
        '\n'
        '\n'
        'def build_extraction_prompt(ocr_text: str) -> str:\n'
        '    """생성형 AI에 전달할 수 있는 짧고 명확한 추출 프롬프트를 만든다."""\n'
        '\n'
        '    schema_text = json.dumps(RECEIPT_SCHEMA, ensure_ascii=False, indent=2)\n'
        '    return f"""역할:\n'
        '당신은 영수증 정보 추출 도우미입니다.\n'
        '\n'
        '목표:\n'
        'OCR 텍스트에서 상호명, 날짜, 품목, 합계를 JSON으로 추출하세요.\n'
        '\n'
        '제약조건:\n'
        '- 원문에 없는 값은 추측하지 말고 null로 반환하세요.\n'
        "- 금액은 쉼표와 '원'을 제외한 정수로 반환하세요.\n"
        '- JSON 외의 설명은 반환하지 마세요.\n'
        '\n'
        'JSON Schema:\n'
        '{schema_text}\n'
        '\n'
        'OCR 텍스트:\n'
        '{ocr_text}\n'
        '"""\n'
        '\n'
        '\n'
        'def _to_int(value: str) -> int:\n'
        '    return int(value.replace(",", ""))\n'
        '\n'
        '\n'
        'def _normalize_date(value: str) -> str:\n'
        '    parts = re.split(r"[-./]", value)\n'
        '    return f"{int(parts[0]):04d}-{int(parts[1]):02d}-{int(parts[2]):02d}"\n'
        '\n'
        '\n'
        'def _find_store_name(lines: list[str]) -> tuple[str | None, int | None]:\n'
        '    ignored = re.compile(\n'
        '        r"^(?:\\[?영수(?:증)?\\]?|거래|사업자|대표|주소|전화|상품명|품명|상\\s*품)",\n'
        '        re.IGNORECASE,\n'
        '    )\n'
        '    for index, line in enumerate(lines, start=1):\n'
        '        candidate = line.lstrip("# ").strip()\n'
        '        if candidate and not ignored.search(candidate):\n'
        '            return candidate, index\n'
        '    return None, None\n'
        '\n'
        '\n'
        'def extract_receipt_from_text(\n'
        '    ocr_text: str,\n'
        '    *,\n'
        '    source_mode: str = "rule_extraction",\n'
        ') -> dict:\n'
        '    """초보자 실습용 최소 규칙으로 영수증 텍스트를 구조화한다.\n'
        '\n'
        '    이 함수는 범용 영수증 AI가 아니다. 한국 영수증에서 자주 보이는 날짜,\n'
        '    합계, `품명 단가 수량 금액` 표기와 수업용 표기를 다루며, 읽지 못한 값은\n'
        '    추측하지 않고 ``None``으로 남긴다.\n'
        '    """\n'
        '\n'
        '    lines = [line.strip() for line in ocr_text.splitlines() if line.strip()]\n'
        '    if not lines:\n'
        '        empty = deepcopy(SAMPLE_RECEIPT)\n'
        '        empty.update(\n'
        '            {\n'
        '                "store_name": None,\n'
        '                "date": None,\n'
        '                "total_amount": None,\n'
        '                "items": [],\n'
        '                "source_mode": source_mode,\n'
        '            }\n'
        '        )\n'
        '        return empty\n'
        '\n'
        '    date_match = re.search(r"\\b(\\d{4}[-./]\\d{1,2}[-./]\\d{1,2})\\b", ocr_text)\n'
        '    total_line_text = next(\n'
        '        (\n'
        '            line\n'
        '            for line in lines\n'
        '            if re.search(\n'
        '                r"(?:합\\s*계|결제\\s*금액|총\\s*액)",\n'
        '                line,\n'
        '                re.IGNORECASE,\n'
        '            )\n'
        '        ),\n'
        '        None,\n'
        '    )\n'
        '    total_candidates = (\n'
        '        re.findall(r"(?<![\\d,])\\d[\\d,]*(?![\\d,])", total_line_text)\n'
        '        if total_line_text\n'
        '        else []\n'
        '    )\n'
        '    total_raw = total_candidates[-1] if total_candidates else None\n'
        '    supply_match = re.search(\n'
        '        r"(?:부가세\\s*)?과세물품가액\\s*[:：]?\\s*(?P<amount>[\\d,]+)",\n'
        '        ocr_text,\n'
        '    )\n'
        '    vat_match = re.search(\n'
        '        r"^부가세(?!\\s*과세물품가액)\\s*[:：]?\\s*(?P<amount>[\\d,]+)",\n'
        '        ocr_text,\n'
        '        re.MULTILINE,\n'
        '    )\n'
        '    item_pattern = re.compile(\n'
        '        r"(?P<name>.+?)\\s+(?P<quantity>\\d+)개\\s*[×x]\\s*"\n'
        '        r"(?P<unit>[\\d,]+)원\\s*=\\s*(?P<line>[\\d,]+)원"\n'
        '    )\n'
        '    markdown_item_pattern = re.compile(\n'
        '        r"^\\|\\s*(?P<name>[^|]+?)\\s*\\|\\s*(?P<quantity>\\d+)\\s*\\|"\n'
        '        r"\\s*(?P<unit>[\\d,]+)원\\s*\\|\\s*(?P<line>[\\d,]+)원\\s*\\|$"\n'
        '    )\n'
        '    receipt_column_pattern = re.compile(\n'
        '        r"^(?P<name>.+?)\\s+(?P<unit>[\\d,]+)\\s+"\n'
        '        r"(?P<quantity>\\d+)\\s+(?P<line>[\\d,]+)\\s*원?$"\n'
        '    )\n'
        '\n'
        '    items = []\n'
        '    item_evidence = []\n'
        '    for line_number, line in enumerate(lines, start=1):\n'
        '        match = item_pattern.search(line)\n'
        '        if not match:\n'
        '            match = markdown_item_pattern.search(line)\n'
        '        if not match:\n'
        '            match = receipt_column_pattern.search(line)\n'
        '        if match:\n'
        '            item = {\n'
        '                "name": match.group("name").strip(),\n'
        '                "quantity": int(match.group("quantity")),\n'
        '                "unit_price": _to_int(match.group("unit")),\n'
        '                "line_total": _to_int(match.group("line")),\n'
        '            }\n'
        '            items.append(item)\n'
        '            item_evidence.append(\n'
        '                {\n'
        '                    "raw_value": line,\n'
        '                    "line": line_number,\n'
        '                    "normalized_value": item,\n'
        '                }\n'
        '            )\n'
        '\n'
        '    store_name, store_line = _find_store_name(lines)\n'
        '    normalized_date = _normalize_date(date_match.group(1)) if date_match else None\n'
        '    total_amount = _to_int(total_raw) if total_raw else None\n'
        '    supply_amount = (\n'
        '        _to_int(supply_match.group("amount")) if supply_match else None\n'
        '    )\n'
        '    vat_amount = _to_int(vat_match.group("amount")) if vat_match else None\n'
        '    total_line = next(\n'
        '        (\n'
        '            index\n'
        '            for index, line in enumerate(lines, start=1)\n'
        '            if total_line_text and line == total_line_text\n'
        '        ),\n'
        '        None,\n'
        '    )\n'
        '\n'
        '    return {\n'
        '        "document_type": "receipt",\n'
        '        "store_name": store_name,\n'
        '        "date": normalized_date,\n'
        '        "total_amount": total_amount,\n'
        '        "items": items,\n'
        '        "adjustments": {\n'
        '            "discount": 0,\n'
        '            "tax": 0,\n'
        '            "service": 0,\n'
        '            "rounding": 0,\n'
        '        },\n'
        '        "tax_breakdown": {\n'
        '            "mode": "included_in_item_prices",\n'
        '            "supply_amount": supply_amount,\n'
        '            "vat": vat_amount,\n'
        '            "payable_total": total_amount,\n'
        '        }\n'
        '        if supply_amount is not None and vat_amount is not None\n'
        '        else None,\n'
        '        "evidence": {\n'
        '            "store_name": {\n'
        '                "raw_value": store_name,\n'
        '                "line": store_line,\n'
        '            },\n'
        '            "date": {\n'
        '                "raw_value": date_match.group(0) if date_match else None,\n'
        '                "line": next(\n'
        '                    (\n'
        '                        index\n'
        '                        for index, line in enumerate(lines, start=1)\n'
        '                        if date_match and date_match.group(0) in line\n'
        '                    ),\n'
        '                    None,\n'
        '                ),\n'
        '            },\n'
        '            "total_amount": {\n'
        '                "raw_value": total_line_text,\n'
        '                "line": total_line,\n'
        '            },\n'
        '            "items": item_evidence,\n'
        '        },\n'
        '        "raw_values": {\n'
        '            "store_name": store_name,\n'
        '            "date": date_match.group(0) if date_match else None,\n'
        '            "total_amount": total_raw,\n'
        '        },\n'
        '        "cleaned_values": {\n'
        '            "store_name": store_name,\n'
        '            "date": normalized_date,\n'
        '            "total_amount": total_amount,\n'
        '        },\n'
        '        "source_mode": source_mode,\n'
        '    }\n'
        '\n'
        '\n'
        'def mock_extract(ocr_text: str) -> dict:\n'
        '    """이전 교재 코드와의 호환을 위한 명시적 합성 fixture 별칭."""\n'
        '\n'
        '    return extract_receipt_from_text(\n'
        '        ocr_text,\n'
        '        source_mode="synthetic_fixture_rule_extraction",\n'
        '    )\n'
        '\n'
        '\n'
        'def validate_schema(data: dict) -> list[str]:\n'
        '    """jsonschema가 있으면 스키마 오류를 쉬운 문장으로 반환한다."""\n'
        '\n'
        '    try:\n'
        '        from jsonschema import Draft202012Validator\n'
        '    except ImportError:\n'
        '        return ["jsonschema가 없어 스키마 검사를 건너뛰었습니다."]\n'
        '\n'
        '    validator = Draft202012Validator(RECEIPT_SCHEMA)\n'
        '    return [error.message for error in validator.iter_errors(data)]\n'
    ),
    'src/ocr.py': (
        '"""2교시: 준비된 결과와 PaddleOCR 3.7 선택 경로."""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'from collections import defaultdict\n'
        'from copy import deepcopy\n'
        'from pathlib import Path\n'
        'from tempfile import TemporaryDirectory\n'
        '\n'
        'from .sample_data import SAMPLE_OCR_RESULT\n'
        '\n'
        '\n'
        'def load_mock_ocr() -> list[dict]:\n'
        '    """API 키나 모델 다운로드 없이 사용할 수업용 OCR 결과를 반환한다."""\n'
        '\n'
        '    return deepcopy(SAMPLE_OCR_RESULT)\n'
        '\n'
        '\n'
        'def reconstruct_spatial_lines(result: list[dict]) -> list[str]:\n'
        '    """OCR 토큰을 페이지·y좌표·x좌표 기준의 읽기 행으로 복원한다.\n'
        '\n'
        '    PaddleOCR는 영수증의 ``품목명 / 단가 / 수량 / 금액`` 한 행을 여러\n'
        '    토큰으로 반환할 수 있다. 단순 줄바꿈으로 합치면 행 관계가 사라지므로,\n'
        '    각 토큰 중심 y좌표가 가까운 것끼리 묶고 x좌표 순서로 정렬한다.\n'
        '    위치가 없는 준비 결과는 입력 순서를 보존한다.\n'
        '    """\n'
        '\n'
        '    positioned_by_page: dict[int, list[dict]] = defaultdict(list)\n'
        '    unpositioned_by_page: dict[int, list[tuple[int, str]]] = defaultdict(list)\n'
        '\n'
        '    for order, item in enumerate(result):\n'
        '        text = " ".join(str(item.get("text", "")).split())\n'
        '        if not text:\n'
        '            continue\n'
        '        page = int(item.get("page") or 1)\n'
        '        points = [\n'
        '            point\n'
        '            for point in (item.get("box") or [])\n'
        '            if isinstance(point, (list, tuple)) and len(point) >= 2\n'
        '        ]\n'
        '        if not points:\n'
        '            unpositioned_by_page[page].append((order, text))\n'
        '            continue\n'
        '        xs = [float(point[0]) for point in points]\n'
        '        ys = [float(point[1]) for point in points]\n'
        '        positioned_by_page[page].append(\n'
        '            {\n'
        '                "text": text,\n'
        '                "x": min(xs),\n'
        '                "y": sum(ys) / len(ys),\n'
        '                "height": max(ys) - min(ys),\n'
        '                "order": order,\n'
        '            }\n'
        '        )\n'
        '\n'
        '    pages = sorted(set(positioned_by_page) | set(unpositioned_by_page))\n'
        '    lines: list[str] = []\n'
        '    for page in pages:\n'
        '        rows: list[dict] = []\n'
        '        for token in sorted(\n'
        '            positioned_by_page[page],\n'
        '            key=lambda value: (value["y"], value["x"], value["order"]),\n'
        '        ):\n'
        '            row = rows[-1] if rows else None\n'
        '            tolerance = (\n'
        '                max(\n'
        '                    12.0,\n'
        '                    min(24.0, max(row["height"], token["height"]) * 0.45),\n'
        '                )\n'
        '                if row\n'
        '                else 12.0\n'
        '            )\n'
        '            if row and abs(token["y"] - row["y"]) <= tolerance:\n'
        '                row["tokens"].append(token)\n'
        '                count = len(row["tokens"])\n'
        '                row["y"] = (row["y"] * (count - 1) + token["y"]) / count\n'
        '                row["height"] = max(row["height"], token["height"])\n'
        '            else:\n'
        '                rows.append(\n'
        '                    {\n'
        '                        "tokens": [token],\n'
        '                        "y": token["y"],\n'
        '                        "height": token["height"],\n'
        '                    }\n'
        '                )\n'
        '\n'
        '        lines.extend(\n'
        '            " ".join(\n'
        '                token["text"]\n'
        '                for token in sorted(\n'
        '                    row["tokens"],\n'
        '                    key=lambda value: (value["x"], value["order"]),\n'
        '                )\n'
        '            )\n'
        '            for row in rows\n'
        '        )\n'
        '        lines.extend(\n'
        '            text\n'
        '            for _, text in sorted(\n'
        '                unpositioned_by_page[page],\n'
        '                key=lambda value: value[0],\n'
        '            )\n'
        '        )\n'
        '    return lines\n'
        '\n'
        '\n'
        'def ocr_text_from_result(result: list[dict]) -> str:\n'
        '    """OCR 결과를 표 행 관계가 보존된 읽기 순서 텍스트로 합친다."""\n'
        '\n'
        '    return "\\n".join(reconstruct_spatial_lines(result))\n'
        '\n'
        '\n'
        'def extract_with_paddleocr(\n'
        '    image_path: str | Path,\n'
        '    *,\n'
        '    lang: str = "korean",\n'
        '    ocr_version: str = "PP-OCRv5",\n'
        ') -> list[dict]:\n'
        '    """PaddleOCR 3.x 결과를 수업의 공통 형식으로 바꾼다.\n'
        '\n'
        '    PP-OCRv6는 현재 한국어 모델을 제공하지 않으므로 한국어 영수증에는\n'
        '    PP-OCRv5 Korean을 명시한다. 설치나 모델 다운로드가 막히면 예외를 그대로\n'
        '    전달한다. 호출하는 쪽에서 오류를 먼저 보여 준 뒤 사용자가 명시적으로\n'
        '    mock 경로를 선택해야 한다.\n'
        '    """\n'
        '\n'
        '    try:\n'
        '        from paddleocr import PaddleOCR\n'
        '    except ImportError as exc:\n'
        '        raise RuntimeError(\n'
        '            "PaddleOCR가 설치되지 않았습니다. \'샘플로 계속\'을 선택하세요."\n'
        '        ) from exc\n'
        '\n'
        '    path = Path(image_path)\n'
        '    if not path.is_file():\n'
        '        raise ValueError(f"이미지 파일을 찾을 수 없습니다: {path}")\n'
        '\n'
        '    pipeline = PaddleOCR(\n'
        '        lang=lang,\n'
        '        ocr_version=ocr_version,\n'
        '        use_doc_orientation_classify=False,\n'
        '        use_doc_unwarping=False,\n'
        '        use_textline_orientation=False,\n'
        '        device="cpu",\n'
        '    )\n'
        '\n'
        '    with TemporaryDirectory(prefix="docai_ocr_") as temp_dir:\n'
        '        image_paths = _prepare_image_paths(path, Path(temp_dir))\n'
        '        result: list[dict] = []\n'
        '        for page_number, image_path in enumerate(image_paths, start=1):\n'
        '            for page_result in pipeline.predict(str(image_path)):\n'
        '                payload = getattr(page_result, "json", page_result)\n'
        '                if callable(payload):\n'
        '                    payload = payload()\n'
        '                page_data = payload.get("res", payload)\n'
        '                texts = page_data.get("rec_texts", [])\n'
        '                scores = page_data.get("rec_scores", [])\n'
        '                boxes = page_data.get("rec_polys", [])\n'
        '                for box, text, confidence in zip(boxes, texts, scores):\n'
        '                    points = box.tolist() if hasattr(box, "tolist") else box\n'
        '                    result.append(\n'
        '                        {\n'
        '                            "page": page_number,\n'
        '                            "box": [\n'
        '                                [int(point[0]), int(point[1])]\n'
        '                                for point in points\n'
        '                            ],\n'
        '                            "text": str(text),\n'
        '                            "confidence": float(confidence),\n'
        '                        }\n'
        '                    )\n'
        '        return result\n'
        '\n'
        '\n'
        'def _prepare_image_paths(path: Path, temp_dir: Path) -> list[Path]:\n'
        '    """이미지는 그대로 사용하고 PDF는 최대 세 페이지를 PNG로 바꾼다."""\n'
        '\n'
        '    if path.suffix.lower() != ".pdf":\n'
        '        return [path]\n'
        '\n'
        '    try:\n'
        '        import fitz\n'
        '    except ImportError as exc:\n'
        '        raise RuntimeError(\n'
        '            "PDF 변환 패키지가 없습니다. PNG 샘플로 계속하세요."\n'
        '        ) from exc\n'
        '\n'
        '    try:\n'
        '        document = fitz.open(path)\n'
        '    except Exception as exc:\n'
        '        raise ValueError("PDF 파일을 열 수 없습니다.") from exc\n'
        '\n'
        '    try:\n'
        '        if document.needs_pass:\n'
        '            raise ValueError("암호가 설정된 PDF는 수업에서 처리하지 않습니다.")\n'
        '        if len(document) > 3:\n'
        '            raise ValueError("수업에서는 PDF를 최대 3페이지만 처리합니다.")\n'
        '\n'
        '        image_paths = []\n'
        '        for page_index, page in enumerate(document):\n'
        '            output_path = temp_dir / f"page_{page_index + 1}.png"\n'
        '            page.get_pixmap(dpi=200, alpha=False).save(output_path)\n'
        '            image_paths.append(output_path)\n'
        '        return image_paths\n'
        '    finally:\n'
        '        document.close()\n'
    ),
    'src/pipeline.py': (
        '"""6~8교시: 작은 처리 함수를 명시적인 기본·mock 경로로 연결한다."""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'from pathlib import Path\n'
        '\n'
        'from .export import receipt_to_xlsx_bytes\n'
        'from .extract import extract_receipt_from_text\n'
        'from .ocr import extract_with_paddleocr, ocr_text_from_result\n'
        'from .sample_data import (\n'
        '    GOLDEN_RECEIPT_OCR_TEXT,\n'
        '    GOLDEN_RECEIPT_VLM_MARKDOWN,\n'
        '    MISSING_STORE_RECEIPT,\n'
        '    SAMPLE_OCR_TEXT,\n'
        '    WRONG_TOTAL_RECEIPT,\n'
        ')\n'
        'from .validate import validate_receipt\n'
        'from .vlm import parse_with_paddleocr_vl, vlm_text_from_result\n'
        '\n'
        '\n'
        'ALLOWED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".pdf"}\n'
        'MAX_FILE_SIZE = 5 * 1024 * 1024\n'
        '\n'
        '\n'
        'def validate_upload(file_path: str | Path | None) -> list[str]:\n'
        '    """수업용 업로드 정책을 확인하고 오류 목록을 반환한다."""\n'
        '\n'
        '    if not file_path:\n'
        '        return ["파일을 선택하세요."]\n'
        '    path = Path(file_path)\n'
        '    if not path.is_file():\n'
        '        return ["업로드 파일을 찾을 수 없습니다."]\n'
        '    if path.suffix.lower() not in ALLOWED_EXTENSIONS:\n'
        '        return ["PNG, JPEG, PDF 파일만 사용할 수 있습니다."]\n'
        '    if path.stat().st_size > MAX_FILE_SIZE:\n'
        '        return ["수업에서는 5MB 이하 파일만 사용합니다."]\n'
        '    if path.suffix.lower() == ".pdf":\n'
        '        try:\n'
        '            import fitz\n'
        '\n'
        '            with fitz.open(path) as document:\n'
        '                if document.page_count != 1:\n'
        '                    return ["필수 실습은 PDF 한 페이지만 처리합니다."]\n'
        '        except ImportError:\n'
        '            pass\n'
        '    return []\n'
        '\n'
        '\n'
        'def process_document(\n'
        '    file_path: str | Path | None = None,\n'
        '    *,\n'
        '    use_sample: bool = False,\n'
        '    processor: str = "ocr",\n'
        '    human_approved: bool = False,\n'
        '    review_record: dict | None = None,\n'
        ') -> dict:\n'
        '    """문서를 처리한다.\n'
        '\n'
        "    `use_sample=True`는 사용자가 '샘플로 계속'을 명시적으로 선택한 경우다.\n"
        '    업로드나 PaddleOCR/PaddleOCR-VL가 실패해도 자동으로 관련 없는 mock\n'
        '    결과를 반환하지 않는다.\n'
        '    """\n'
        '\n'
        '    if use_sample:\n'
        '        if processor == "ocr":\n'
        '            ocr_text = GOLDEN_RECEIPT_OCR_TEXT\n'
        '            extraction_mode = "prepared_fixture_rule_extraction"\n'
        '            mode = "PREPARED REPLAY — 공개 한국 영수증의 검수된 OCR 텍스트"\n'
        '            target_technology = "PaddleOCR Korean"\n'
        '        elif processor == "vlm":\n'
        '            ocr_text = GOLDEN_RECEIPT_VLM_MARKDOWN\n'
        '            extraction_mode = "prepared_vlm_structure_fixture_rule_extraction"\n'
        '            mode = "PREPARED REPLAY — 공개 한국 영수증의 VLM 구조 시연 fixture"\n'
        '            target_technology = "PaddleOCR-VL-1.6"\n'
        '        else:\n'
        '            return {\n'
        '                "ok": False,\n'
        '                "status": "처리 방식 오류",\n'
        '                "errors": ["processor는 \'ocr\' 또는 \'vlm\'이어야 합니다."],\n'
        '                "can_continue_with_sample": True,\n'
        '            }\n'
        '        provenance = {\n'
        '            "fixture_type": (\n'
        '                "human_verified_transcription_fixture"\n'
        '                if processor == "ocr"\n'
        '                else "prepared_demonstration_fixture"\n'
        '            ),\n'
        '            "input_file": "taebaek_restaurant_2025_redacted.png",\n'
        '            "input_sha256": (\n'
        '                "19227c7298a16ee69bef2d7bed65826b8a1cba5389375e4ae77d02005362641f"\n'
        '            ),\n'
        '            "engine": "not_executed",\n'
        '            "engine_version": "not_applicable",\n'
        '            "target_technology": target_technology,\n'
        '            "recorded_at": "2026-07-28",\n'
        '            "reviewer": "course maintainer",\n'
        '            "disclaimer": (\n'
        '                "현재 실행에서 모델을 호출한 결과가 아닙니다. "\n'
        '                "VLM 경로는 구조와 검증 활동을 위한 시연 fixture입니다."\n'
        '                if processor == "vlm"\n'
        '                else "현재 실행에서 모델을 호출한 결과가 아닙니다."\n'
        '            ),\n'
        '        }\n'
        '    else:\n'
        '        errors = validate_upload(file_path)\n'
        '        if errors:\n'
        '            return {\n'
        '                "ok": False,\n'
        '                "status": "입력 오류",\n'
        '                "errors": errors,\n'
        '                "can_continue_with_sample": True,\n'
        '            }\n'
        '        try:\n'
        '            if processor == "ocr":\n'
        '                ocr_result = extract_with_paddleocr(Path(file_path))\n'
        '                ocr_text = ocr_text_from_result(ocr_result)\n'
        '                mode = (\n'
        '                    "LIVE PaddleOCR 3.7 / PP-OCRv5 Korean + 규칙 추출"\n'
        '                )\n'
        '                extraction_mode = "live_ocr_rule_extraction"\n'
        '                provenance = {\n'
        '                    "fixture_type": "live_inference",\n'
        '                    "input_file": Path(file_path).name,\n'
        '                    "engine": "PaddleOCR",\n'
        '                    "engine_version": "3.7 / PP-OCRv5 Korean",\n'
        '                    "recorded_at": "",\n'
        '                    "reviewer": "",\n'
        '                    "disclaimer": "",\n'
        '                }\n'
        '            elif processor == "vlm":\n'
        '                vlm_result = parse_with_paddleocr_vl(Path(file_path))\n'
        '                ocr_text = vlm_text_from_result(vlm_result)\n'
        '                mode = "LIVE PaddleOCR-VL 1.6 + 규칙 추출"\n'
        '                extraction_mode = "live_vlm_rule_extraction"\n'
        '                provenance = {\n'
        '                    "fixture_type": "live_inference",\n'
        '                    "input_file": Path(file_path).name,\n'
        '                    "engine": "PaddleOCR-VL",\n'
        '                    "engine_version": "1.6",\n'
        '                    "recorded_at": "",\n'
        '                    "reviewer": "",\n'
        '                    "disclaimer": "",\n'
        '                }\n'
        '            else:\n'
        '                return {\n'
        '                    "ok": False,\n'
        '                    "status": "처리 방식 오류",\n'
        '                    "errors": ["processor는 \'ocr\' 또는 \'vlm\'이어야 합니다."],\n'
        '                    "can_continue_with_sample": True,\n'
        '                }\n'
        '        except Exception as exc:\n'
        '            return {\n'
        '                "ok": False,\n'
        '                "status": "문서 처리 오류",\n'
        '                "errors": [str(exc)],\n'
        '                "can_continue_with_sample": True,\n'
        '            }\n'
        '\n'
        '    extracted = extract_receipt_from_text(\n'
        '        ocr_text,\n'
        '        source_mode=extraction_mode,\n'
        '    )\n'
        '    extracted["provenance"] = provenance\n'
        '    validation = validate_receipt(extracted)\n'
        '    if review_record is None:\n'
        '        review_record = {\n'
        '            "decision": "APPROVED" if human_approved else "PENDING",\n'
        '            "reviewer": "learner" if human_approved else "",\n'
        '            "reviewed_at": "",\n'
        '            "note": "",\n'
        '        }\n'
        '    decision = review_record.get("decision", "PENDING")\n'
        '    review_status = (\n'
        '        decision\n'
        '        if validation["valid"] and decision in {"APPROVED", "CHANGED"}\n'
        '        else "PENDING_REVIEW"\n'
        '        if validation["valid"]\n'
        '        else "BLOCKED_BY_VALIDATION"\n'
        '    )\n'
        '    xlsx_bytes = (\n'
        '        receipt_to_xlsx_bytes(\n'
        '            extracted,\n'
        '            source_text=ocr_text,\n'
        '            review_status=review_status,\n'
        '            review_record=review_record,\n'
        '        )\n'
        '        if review_status in {"APPROVED", "CHANGED"}\n'
        '        else None\n'
        '    )\n'
        '\n'
        '    return {\n'
        '        "ok": True,\n'
        '        "status": mode,\n'
        '        "ocr_text": ocr_text,\n'
        '        "data": extracted,\n'
        '        "validation": validation,\n'
        '        "review_status": review_status,\n'
        '        "review_record": review_record,\n'
        '        "xlsx_bytes": xlsx_bytes,\n'
        '    }\n'
        '\n'
        '\n'
        'def run_smoke_test() -> dict[str, bool]:\n'
        '    """8교시에서 한 번 호출해 정상·누락·합계 오류·mock 경로를 점검한다."""\n'
        '\n'
        '    sample_result = process_document(use_sample=True)\n'
        '    missing_result = validate_receipt(MISSING_STORE_RECEIPT)\n'
        '    total_result = validate_receipt(WRONG_TOTAL_RECEIPT)\n'
        '    return {\n'
        '        "mock_path_works": bool(sample_result.get("ok")),\n'
        '        "normal_result_is_valid": bool(\n'
        '            sample_result.get("validation", {}).get("valid")\n'
        '        ),\n'
        '        "missing_required_is_blocked": not missing_result["valid"],\n'
        '        "wrong_total_is_blocked": not total_result["valid"],\n'
        '    }\n'
    ),
    'src/sample_data.py': (
        '"""2~8교시와 데모 앱에서 사용하는 골든 영수증·복구 데이터."""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'from copy import deepcopy\n'
        '\n'
        '\n'
        'SAMPLE_OCR_TEXT = """샘플문구점\n'
        '거래일자: 2026-07-27\n'
        '연필 2개 × 1,000원 = 2,000원\n'
        '노트 1개 × 3,000원 = 3,000원\n'
        '합계: 5,000원\n'
        '"""\n'
        '\n'
        'GOLDEN_RECEIPT_OCR_TEXT = """이태리집\n'
        '거래일시 2025-10-04 12:33:37\n'
        '페퍼로니 앤 치즈 29,000 1 29,000\n'
        '토마토 파스타 14,000 1 14,000\n'
        '수제 돈가스 13,000 1 13,000\n'
        '새우 칠리치 필라 14,000 1 14,000\n'
        '콜라 2,000 3 6,000\n'
        '합계 금액 76,000\n'
        '부가세 과세물품가액 69,094\n'
        '부가세 6,906\n'
        '"""\n'
        '\n'
        'GOLDEN_RECEIPT_VLM_MARKDOWN = """# 이태리집\n'
        '\n'
        '> **PREPARED VLM STRUCTURE FIXTURE** — 현재 실행에서 VLM을 호출한 결과가 아닙니다.\n'
        '\n'
        '거래일시: 2025-10-04 12:33:37\n'
        '\n'
        '| 품목 | 수량 | 단가 | 금액 |\n'
        '| --- | ---: | ---: | ---: |\n'
        '| 페퍼로니 앤 치즈 | 1 | 29,000원 | 29,000원 |\n'
        '| 토마토 파스타 | 1 | 14,000원 | 14,000원 |\n'
        '| 수제 돈가스 | 1 | 13,000원 | 13,000원 |\n'
        '| 새우 칠리치 필라 | 1 | 14,000원 | 14,000원 |\n'
        '| 콜라 | 3 | 2,000원 | 6,000원 |\n'
        '\n'
        '**합계: 76,000원**\n'
        '\n'
        '부가세 과세물품가액 69,094\n'
        '부가세 6,906\n'
        '"""\n'
        '\n'
        'GOLDEN_RECEIPT_VLM_RESULT = {\n'
        '    "target_technology": "PaddleOCR-VL-1.6",\n'
        '    "executed_model": None,\n'
        '    "source_mode": "prepared_vlm_structure_fixture",\n'
        '    "provenance": {\n'
        '        "fixture_type": "prepared_demonstration_fixture",\n'
        '        "input_file": "taebaek_restaurant_2025_redacted.png",\n'
        '        "engine": "not_executed",\n'
        '        "engine_version": "not_applicable",\n'
        '        "target_technology": "PaddleOCR-VL-1.6",\n'
        '        "created_by": "course maintainer",\n'
        '        "disclaimer": "현재 실행에서 VLM을 호출한 결과가 아닙니다.",\n'
        '    },\n'
        '    "pages": [\n'
        '        {\n'
        '            "page": 1,\n'
        '            "markdown": GOLDEN_RECEIPT_VLM_MARKDOWN,\n'
        '            "blocks": [\n'
        '                {"label": "title", "content": "이태리집", "order": 1},\n'
        '                {\n'
        '                    "label": "text",\n'
        '                    "content": "거래일시: 2025-10-04 12:33:37",\n'
        '                    "order": 2,\n'
        '                },\n'
        '                {\n'
        '                    "label": "table",\n'
        '                    "content": "품목·수량·단가·금액 5행",\n'
        '                    "order": 3,\n'
        '                },\n'
        '                {"label": "text", "content": "합계: 76,000원", "order": 4},\n'
        '            ],\n'
        '        }\n'
        '    ],\n'
        '}\n'
        '\n'
        'SAMPLE_OCR_RESULT = [\n'
        '    {\n'
        '        "box": [[80, 70], [330, 70], [330, 125], [80, 125]],\n'
        '        "text": "샘플문구점",\n'
        '        "confidence": 0.98,\n'
        '    },\n'
        '    {\n'
        '        "box": [[80, 160], [500, 160], [500, 205], [80, 205]],\n'
        '        "text": "거래일자: 2026-07-27",\n'
        '        "confidence": 0.97,\n'
        '    },\n'
        '    {\n'
        '        "box": [[80, 270], [650, 270], [650, 315], [80, 315]],\n'
        '        "text": "연필 2개 × 1,000원 = 2,000원",\n'
        '        "confidence": 0.93,\n'
        '    },\n'
        '    {\n'
        '        "box": [[80, 340], [650, 340], [650, 385], [80, 385]],\n'
        '        "text": "노트 1개 × 3,000원 = 3,000원",\n'
        '        "confidence": 0.95,\n'
        '    },\n'
        '    {\n'
        '        "box": [[80, 465], [440, 465], [440, 520], [80, 520]],\n'
        '        "text": "합계: 5,000원",\n'
        '        "confidence": 0.99,\n'
        '    },\n'
        ']\n'
        '\n'
        'SAMPLE_VLM_MARKDOWN = """# 샘플문구점\n'
        '\n'
        '> **PREPARED SYNTHETIC FIXTURE** — 현재 실행에서 VLM을 호출한 결과가 아닙니다.\n'
        '\n'
        '거래일자: 2026-07-27\n'
        '\n'
        '| 품목 | 수량 | 단가 | 금액 |\n'
        '| --- | ---: | ---: | ---: |\n'
        '| 연필 | 2 | 1,000원 | 2,000원 |\n'
        '| 노트 | 1 | 3,000원 | 3,000원 |\n'
        '\n'
        '**합계: 5,000원**\n'
        '"""\n'
        '\n'
        'SAMPLE_VLM_RESULT = {\n'
        '    "target_technology": "PaddleOCR-VL-1.6",\n'
        '    "executed_model": None,\n'
        '    "source_mode": "synthetic_fixture",\n'
        '    "provenance": {\n'
        '        "fixture_type": "synthetic_fixture",\n'
        '        "input_file": "receipt_sample.png",\n'
        '        "created_by": "course generator",\n'
        '        "disclaimer": "현재 실행에서 VLM을 호출한 결과가 아닙니다.",\n'
        '    },\n'
        '    "pages": [\n'
        '        {\n'
        '            "page": 1,\n'
        '            "markdown": SAMPLE_VLM_MARKDOWN,\n'
        '            "blocks": [\n'
        '                {"label": "title", "content": "샘플문구점", "order": 1},\n'
        '                {"label": "text", "content": "거래일자: 2026-07-27", "order": 2},\n'
        '                {\n'
        '                    "label": "table",\n'
        '                    "content": "| 품목 | 수량 | 단가 | 금액 |",\n'
        '                    "order": 3,\n'
        '                },\n'
        '                {"label": "text", "content": "합계: 5,000원", "order": 4},\n'
        '            ],\n'
        '        }\n'
        '    ],\n'
        '}\n'
        '\n'
        'SAMPLE_RECEIPT = {\n'
        '    "document_type": "receipt",\n'
        '    "store_name": "샘플문구점",\n'
        '    "date": "2026-07-27",\n'
        '    "total_amount": 5000,\n'
        '    "items": [\n'
        '        {"name": "연필", "quantity": 2, "unit_price": 1000, "line_total": 2000},\n'
        '        {"name": "노트", "quantity": 1, "unit_price": 3000, "line_total": 3000},\n'
        '    ],\n'
        '    "adjustments": {\n'
        '        "discount": 0,\n'
        '        "tax": 0,\n'
        '        "service": 0,\n'
        '        "rounding": 0,\n'
        '    },\n'
        '    "evidence": {\n'
        '        "store_name": {"raw_value": "샘플문구점", "line": 1},\n'
        '        "date": {"raw_value": "거래일자: 2026-07-27", "line": 2},\n'
        '        "total_amount": {"raw_value": "합계: 5,000원", "line": 5},\n'
        '    },\n'
        '    "raw_values": {\n'
        '        "store_name": "샘플문구점",\n'
        '        "date": "2026-07-27",\n'
        '        "total_amount": "5,000원",\n'
        '    },\n'
        '    "cleaned_values": {\n'
        '        "store_name": "샘플문구점",\n'
        '        "date": "2026-07-27",\n'
        '        "total_amount": 5000,\n'
        '    },\n'
        '    "source_mode": "synthetic_fixture_rule_extraction",\n'
        '}\n'
        '\n'
        'GOLDEN_RECEIPT = {\n'
        '    "document_type": "receipt",\n'
        '    "store_name": "이태리집",\n'
        '    "date": "2025-10-04",\n'
        '    "total_amount": 76000,\n'
        '    "items": [\n'
        '        {\n'
        '            "name": "페퍼로니 앤 치즈",\n'
        '            "quantity": 1,\n'
        '            "unit_price": 29000,\n'
        '            "line_total": 29000,\n'
        '        },\n'
        '        {\n'
        '            "name": "토마토 파스타",\n'
        '            "quantity": 1,\n'
        '            "unit_price": 14000,\n'
        '            "line_total": 14000,\n'
        '        },\n'
        '        {\n'
        '            "name": "수제 돈가스",\n'
        '            "quantity": 1,\n'
        '            "unit_price": 13000,\n'
        '            "line_total": 13000,\n'
        '        },\n'
        '        {\n'
        '            "name": "새우 칠리치 필라",\n'
        '            "quantity": 1,\n'
        '            "unit_price": 14000,\n'
        '            "line_total": 14000,\n'
        '        },\n'
        '        {\n'
        '            "name": "콜라",\n'
        '            "quantity": 3,\n'
        '            "unit_price": 2000,\n'
        '            "line_total": 6000,\n'
        '        },\n'
        '    ],\n'
        '    "adjustments": {\n'
        '        "discount": 0,\n'
        '        "tax": 0,\n'
        '        "service": 0,\n'
        '        "rounding": 0,\n'
        '    },\n'
        '    "tax_breakdown": {\n'
        '        "mode": "included_in_item_prices",\n'
        '        "supply_amount": 69094,\n'
        '        "vat": 6906,\n'
        '        "payable_total": 76000,\n'
        '    },\n'
        '    "evidence": {\n'
        '        "store_name": {"raw_value": "이태리집", "line": 1},\n'
        '        "date": {\n'
        '            "raw_value": "거래일시 2025-10-04 12:33:37",\n'
        '            "line": 2,\n'
        '        },\n'
        '        "total_amount": {"raw_value": "합계 금액 76,000", "line": 8},\n'
        '    },\n'
        '    "raw_values": {\n'
        '        "store_name": "이태리집",\n'
        '        "date": "2025-10-04 12:33:37",\n'
        '        "total_amount": "76,000",\n'
        '    },\n'
        '    "cleaned_values": {\n'
        '        "store_name": "이태리집",\n'
        '        "date": "2025-10-04",\n'
        '        "total_amount": 76000,\n'
        '    },\n'
        '    "source_mode": "prepared_fixture_rule_extraction",\n'
        '}\n'
        '\n'
        'MISSING_STORE_RECEIPT = {\n'
        '    **SAMPLE_RECEIPT,\n'
        '    "store_name": None,\n'
        '    "items": deepcopy(SAMPLE_RECEIPT["items"]),\n'
        '}\n'
        '\n'
        'WRONG_TOTAL_RECEIPT = {\n'
        '    **SAMPLE_RECEIPT,\n'
        '    "total_amount": 6000,\n'
        '    "items": deepcopy(SAMPLE_RECEIPT["items"]),\n'
        '}\n'
        '\n'
        '\n'
        'def sample_receipt() -> dict:\n'
        '    """호출한 코드가 원본 상수를 바꾸지 않도록 복사본을 반환한다."""\n'
        '\n'
        '    return deepcopy(SAMPLE_RECEIPT)\n'
    ),
    'src/validate.py': (
        '"""7교시: 영수증 결과의 필수값과 합계를 확인한다."""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'from datetime import date\n'
        'from typing import Any\n'
        '\n'
        '\n'
        'def _is_iso_date(value: Any) -> bool:\n'
        '    if not isinstance(value, str):\n'
        '        return False\n'
        '    try:\n'
        '        date.fromisoformat(value)\n'
        '        return True\n'
        '    except ValueError:\n'
        '        return False\n'
        '\n'
        '\n'
        'def validate_receipt(data: dict) -> dict:\n'
        '    """초보자 수업에 필요한 최소 영수증 규칙만 검사한다."""\n'
        '\n'
        '    warnings: list[str] = []\n'
        '    errors: list[str] = []\n'
        '\n'
        '    for field in ("store_name", "date", "total_amount", "items"):\n'
        '        if data.get(field) in (None, "", []):\n'
        '            errors.append(f"필수값 누락: {field}")\n'
        '\n'
        '    receipt_date = data.get("date")\n'
        '    if receipt_date and not _is_iso_date(receipt_date):\n'
        '        errors.append("date는 YYYY-MM-DD 형식이어야 합니다.")\n'
        '    elif receipt_date and date.fromisoformat(receipt_date) > date.today():\n'
        '        warnings.append("date가 오늘보다 미래입니다. 원본 날짜를 확인하세요.")\n'
        '\n'
        '    total_amount = data.get("total_amount")\n'
        '    if total_amount is not None and (\n'
        '        isinstance(total_amount, bool)\n'
        '        or not isinstance(total_amount, int)\n'
        '        or total_amount < 0\n'
        '    ):\n'
        '        errors.append("total_amount는 0 이상의 정수여야 합니다.")\n'
        '\n'
        '    items = data.get("items") or []\n'
        '    if items:\n'
        '        line_total_sum = 0\n'
        '        for index, item in enumerate(items, start=1):\n'
        '            if not item.get("name"):\n'
        '                errors.append(f"{index}번째 품목 이름이 비어 있습니다.")\n'
        '            for field in ("quantity", "unit_price", "line_total"):\n'
        '                value = item.get(field)\n'
        '                if (\n'
        '                    isinstance(value, bool)\n'
        '                    or not isinstance(value, int)\n'
        '                    or value < 0\n'
        '                ):\n'
        '                    errors.append(\n'
        '                        f"{index}번째 품목의 {field}는 0 이상의 정수여야 합니다."\n'
        '                    )\n'
        '            quantity = item.get("quantity")\n'
        '            unit_price = item.get("unit_price")\n'
        '            line_total = item.get("line_total")\n'
        '            if (\n'
        '                isinstance(quantity, int)\n'
        '                and not isinstance(quantity, bool)\n'
        '                and isinstance(unit_price, int)\n'
        '                and not isinstance(unit_price, bool)\n'
        '                and isinstance(line_total, int)\n'
        '                and not isinstance(line_total, bool)\n'
        '            ):\n'
        '                line_total_sum += line_total\n'
        '                if quantity * unit_price != line_total:\n'
        '                    errors.append(\n'
        '                        f"{index}번째 품목: 수량×단가와 품목 금액이 다릅니다."\n'
        '                    )\n'
        '\n'
        '        adjustments = data.get("adjustments") or {}\n'
        '        adjustment_total = sum(\n'
        '            value\n'
        '            for value in (\n'
        '                -adjustments.get("discount", 0),\n'
        '                adjustments.get("tax", 0),\n'
        '                adjustments.get("service", 0),\n'
        '                adjustments.get("rounding", 0),\n'
        '            )\n'
        '            if isinstance(value, int) and not isinstance(value, bool)\n'
        '        )\n'
        '        expected_total = line_total_sum + adjustment_total\n'
        '        if (\n'
        '            isinstance(total_amount, int)\n'
        '            and not isinstance(total_amount, bool)\n'
        '            and expected_total != total_amount\n'
        '        ):\n'
        '            errors.append(\n'
        '                f"품목·조정 후 합계 {expected_total:,}원과 총액 "\n'
        '                f"{total_amount:,}원이 다릅니다."\n'
        '            )\n'
        '\n'
        '    tax_breakdown = data.get("tax_breakdown")\n'
        '    if tax_breakdown and tax_breakdown.get("mode") == "included_in_item_prices":\n'
        '        supply_amount = tax_breakdown.get("supply_amount")\n'
        '        vat = tax_breakdown.get("vat")\n'
        '        payable_total = tax_breakdown.get("payable_total")\n'
        '        if not all(\n'
        '            isinstance(value, int) and not isinstance(value, bool)\n'
        '            for value in (supply_amount, vat, payable_total)\n'
        '        ):\n'
        '            errors.append("포함세액 내역은 정수 금액이어야 합니다.")\n'
        '        elif supply_amount + vat != payable_total:\n'
        '            errors.append("공급가액과 포함 부가세의 합이 결제금액과 다릅니다.")\n'
        '        elif payable_total != total_amount:\n'
        '            errors.append("포함세액 내역의 결제금액과 영수증 총액이 다릅니다.")\n'
        '        if (data.get("adjustments") or {}).get("tax", 0) != 0:\n'
        '            errors.append(\n'
        '                "포함 부가세를 adjustments.tax에 다시 더하면 이중 계산됩니다."\n'
        '            )\n'
        '\n'
        '    evidence = data.get("evidence") or {}\n'
        '    for field in ("store_name", "date", "total_amount"):\n'
        '        if data.get(field) not in (None, "") and not evidence.get(field):\n'
        '            warnings.append(f"{field}의 원본 근거가 없습니다.")\n'
        '\n'
        '    return {\n'
        '        "valid": not errors,\n'
        '        "warnings": warnings,\n'
        '        "errors": errors,\n'
        '    }\n'
    ),
    'src/vlm.py': (
        '"""4교시: PaddleOCR-VL 1.6 문서 파싱 선택 경로."""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'from copy import deepcopy\n'
        'from pathlib import Path\n'
        '\n'
        'from .sample_data import SAMPLE_VLM_RESULT\n'
        '\n'
        '\n'
        'def load_mock_vlm() -> dict:\n'
        '    """모델 다운로드 없이 사용할 합성 VLM 문서 파싱 결과를 반환한다."""\n'
        '\n'
        '    return deepcopy(SAMPLE_VLM_RESULT)\n'
        '\n'
        '\n'
        'def vlm_text_from_result(result: dict) -> str:\n'
        '    """페이지별 Markdown을 정보 추출용 텍스트로 합친다."""\n'
        '\n'
        '    return "\\n\\n".join(\n'
        '        page.get("markdown", "")\n'
        '        for page in result.get("pages", [])\n'
        '        if page.get("markdown")\n'
        '    )\n'
        '\n'
        '\n'
        'def parse_with_paddleocr_vl(\n'
        '    document_path: str | Path,\n'
        '    *,\n'
        '    engine: str = "transformers",\n'
        ') -> dict:\n'
        '    """PaddleOCR-VL 1.6 전체 파이프라인으로 문서를 파싱한다."""\n'
        '\n'
        '    try:\n'
        '        from paddleocr import PaddleOCRVL\n'
        '    except ImportError as exc:\n'
        '        raise RuntimeError(\n'
        '            "PaddleOCR-VL 의존성이 없습니다. \'샘플로 계속\'을 선택하세요."\n'
        '        ) from exc\n'
        '\n'
        '    path = Path(document_path)\n'
        '    if not path.is_file():\n'
        '        raise ValueError(f"문서 파일을 찾을 수 없습니다: {path}")\n'
        '\n'
        '    pipeline = PaddleOCRVL(\n'
        '        pipeline_version="v1.6",\n'
        '        engine=engine,\n'
        '        use_doc_orientation_classify=False,\n'
        '        use_doc_unwarping=False,\n'
        '    )\n'
        '\n'
        '    pages = []\n'
        '    for page_number, page_result in enumerate(\n'
        '        pipeline.predict(str(path)),\n'
        '        start=1,\n'
        '    ):\n'
        '        payload = getattr(page_result, "json", {})\n'
        '        if callable(payload):\n'
        '            payload = payload()\n'
        '        page_data = payload.get("res", payload)\n'
        '\n'
        '        markdown_payload = getattr(page_result, "markdown", {}) or {}\n'
        '        if callable(markdown_payload):\n'
        '            markdown_payload = markdown_payload()\n'
        '        markdown_text = (\n'
        '            markdown_payload.get("markdown_texts")\n'
        '            or markdown_payload.get("text")\n'
        '            or page_data.get("markdown", "")\n'
        '        )\n'
        '        if isinstance(markdown_text, list):\n'
        '            markdown_text = "\\n\\n".join(map(str, markdown_text))\n'
        '\n'
        '        blocks = []\n'
        '        for block in page_data.get("parsing_res_list", []):\n'
        '            blocks.append(\n'
        '                {\n'
        '                    "label": block.get("block_label"),\n'
        '                    "content": block.get("block_content"),\n'
        '                    "order": block.get("block_order"),\n'
        '                }\n'
        '            )\n'
        '        pages.append(\n'
        '            {\n'
        '                "page": page_number,\n'
        '                "markdown": str(markdown_text or ""),\n'
        '                "blocks": blocks,\n'
        '            }\n'
        '        )\n'
        '\n'
        '    return {\n'
        '        "model": "PaddleOCR-VL-1.6",\n'
        '        "source_mode": "live_vlm",\n'
        '        "pages": pages,\n'
        '    }\n'
    ),
}

final_app_dir = OUTPUT_DIR / "final_document_ai_app"
for relative_path, source in packaged_sources.items():
    target = final_app_dir / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")

final_app_path = final_app_dir / "app.py"
archive_base = OUTPUT_DIR / "final_document_ai_app"
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=final_app_dir,
    )
)
print("최종 앱:", final_app_path)
print("앱 전체 코드:", archive_path)

complete_lab_step(10, 12, '최종 앱 경로와 ZIP 파일 경로가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `AppTest`가 오류값 저장 차단, 정상값 복구, 사람 승인, Excel 다운로드를 버튼 조작으로 검사합니다. 최종 통합 테스트입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(11, 12, '최종 앱 자동 동작 검사', '수정·재검증·승인 전 차단·승인 후 다운로드를 검사합니다.', '`FINAL APP PASS`가 표시되어야 합니다.', '`AppTest`가 오류값 저장 차단, 정상값 복구, 사람 승인, Excel 다운로드를 버튼 조작으로 검사합니다. 최종 통합 테스트입니다.')

import sys
from streamlit.testing.v1 import AppTest

sys.path.insert(0, str(final_app_dir))
final_test = AppTest.from_file(str(final_app_path)).run(timeout=30)
assert not final_test.exception
final_test.button(key="run_sample").click().run(timeout=30)
assert not final_test.exception
assert any(
    "원본 대조 후 수정" in item.value
    for item in final_test.subheader
)
assert len(final_test.get("download_button")) == 0
final_test.checkbox(key="review_complete").check().run(timeout=30)
assert not final_test.exception
assert len(final_test.get("download_button")) == 1
print("FINAL APP PASS: 수정 표·재검증·승인·Excel 다운로드")
download_artifact(archive_path)

complete_lab_step(11, 12, '`FINAL APP PASS`가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# 최종 Streamlit 앱을 Colab iframe으로 열어 전체 흐름을 직접 조작합니다. 자동검증에서는 서버 화면만 생략합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(12, 12, '최종 앱 직접 조작', 'Colab 안에서 전체 Document AI 흐름을 직접 수행합니다.', '앱 화면 또는 검증 모드 생략 안내를 확인합니다.', '최종 Streamlit 앱을 Colab iframe으로 열어 전체 흐름을 직접 조작합니다. 자동검증에서는 서버 화면만 생략합니다.')

# 선택 실습 · 녹화에서는 이 셀로 실제 화면을 엽니다.
# AppTest가 필수 검증이며, 미리보기에는 공개 비식별 샘플만 사용합니다.
if not VALIDATION_MODE:
    import subprocess
    import time
    import urllib.request

    preview_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run",
            str(final_app_path),
            "--server.port", "8507",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    for _ in range(20):
        try:
            urllib.request.urlopen(
                "http://127.0.0.1:8507/_stcore/health",
                timeout=1,
            )
            break
        except Exception:
            time.sleep(0.5)
    try:
        from google.colab import output
        print("아래 화면에서 직접 버튼과 입력값을 조작하세요.")
        output.serve_kernel_port_as_iframe(8507, height=760)
    except Exception as exc:
        print("Colab 미리보기를 열지 못했습니다:", exc)
        print("AppTest 결과와 app 파일로 계속합니다.")
else:
    print("검증 모드: 대화형 Streamlit 미리보기 생략")

complete_lab_step(12, 12, '앱 화면 또는 검증 모드 생략 안내를 확인합니다.')
